# 09 · Explore the corpus

> **Run order.** This notebook is step - of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Read-only. Safe to run any time to see what is loaded, and the fastest way to
convince yourself the structured and unstructured halves really do join.

In [1]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from sqlalchemy import text
from analyst.db import session_scope

def q(sql: str) -> pd.DataFrame:
    with session_scope() as s:
        return pd.DataFrame(s.execute(text(sql)).mappings().all())

q('''
SELECT 'companies' AS table, count(*) AS rows FROM companies
UNION ALL SELECT 'documents', count(*) FROM documents
UNION ALL SELECT 'elements',  count(*) FROM elements
UNION ALL SELECT 'facts',     count(*) FROM facts
UNION ALL SELECT 'prices',    count(*) FROM prices
ORDER BY 1
''')

,table,rows
0,companies,12
1,documents,6
2,elements,46241
3,facts,8532
4,prices,14879


## Reporting currency is not the quote currency

In [2]:
q("SELECT financial_currency AS ccy, count(*) AS n, string_agg(ticker, ', ') AS tickers "
  "FROM companies GROUP BY 1 ORDER BY 2 DESC")

,ccy,n,tickers
0,INR,11,"TCS, WIPRO, HDFCBANK, ICICIBANK, BAJFINANCE, R..."
1,USD,1,INFY


## Elements per document

In [3]:
q('''
SELECT d.ticker, d.fiscal_year AS fy, d.n_pages AS pages,
       count(*) FILTER (WHERE e.type='text')    AS text,
       count(*) FILTER (WHERE e.type='heading') AS heading,
       count(*) FILTER (WHERE e.type='table')   AS "table",
       count(*) FILTER (WHERE e.type='figure')  AS figure
FROM documents d JOIN elements e ON e.document_id = d.document_id
GROUP BY 1,2,3 ORDER BY 1,2
''')

,ticker,fy,pages,text,heading,table,figure
0,HDFCBANK,2025,590,9894,1915,327,167
1,ICICIBANK,2024,341,4335,648,323,30
2,ICICIBANK,2025,341,4470,781,323,60
3,RELIANCE,2025,146,6116,1118,266,39
4,SUNPHARMA,2024,312,6544,851,365,54
5,SUNPHARMA,2025,326,6316,858,373,68


## Latest close, per ticker

`DISTINCT ON` gives each ticker its *own* latest session. Keying off a global `max(trade_date)` returns NULL for any ticker whose last bar was dropped as NaN — a query bug that looks exactly like a data bug.

In [4]:
q('''
SELECT DISTINCT ON (p.ticker) p.ticker, p.trade_date AS latest_session,
       round(p.close, 2) AS close
FROM prices p ORDER BY p.ticker, p.trade_date DESC
''')

,ticker,latest_session,close
0,BAJFINANCE,2026-09-04,1060.50
1,CIPLA,2026-09-04,1385.00
2,DRREDDY,2026-09-03,1155.00
3,HDFCBANK,2026-09-04,712.10
4,ICICIBANK,2026-09-04,1423.20
5,INFY,2026-09-04,1130.00
6,NTPC,2026-09-04,332.50
7,ONGC,2026-09-04,234.65
8,RELIANCE,2026-09-04,1322.00
9,SUNPHARMA,2026-09-04,1899.00


## The join, proved

Take a value we know to be true and find it on a real page of a real report. This is notebook 06's generator in miniature.

In [5]:
from datetime import date
from decimal import Decimal
from sqlalchemy import select
from analyst.models import Document, ElementRow, Fact
from analyst.numfmt import find_value

TICKER, FY = "SUNPHARMA", 2025
CONCEPTS = ("Total Revenue", "Net Income", "Gross Profit", "Operating Income")

with session_scope() as s:
    doc = s.execute(select(Document).where(Document.ticker == TICKER,
                                           Document.fiscal_year == FY)).scalar_one()
    facts = s.execute(select(Fact.concept, Fact.value).where(Fact.ticker == TICKER)
                      .where(Fact.concept.in_(CONCEPTS))
                      .where(Fact.period_end == date(FY, 3, 31))).all()
    elements = s.execute(select(ElementRow.element_id, ElementRow.page, ElementRow.type,
                                ElementRow.text)
                         .where(ElementRow.document_id == doc.document_id)
                         .where(ElementRow.text.is_not(None))
                         .order_by(ElementRow.page, ElementRow.seq)).all()

rows = []
for concept, value in facts:
    hit = next(((eid, page, etype, m) for eid, page, etype, t in elements
                if (m := find_value(Decimal(value), t or ""))), None)
    rows.append({
        "concept": concept,
        "oracle_cr": f"{Decimal(value) / Decimal(10**7):,.0f}",
        "found_as": hit[3] if hit else "-- DISCARDED --",
        "page": hit[1] if hit else None,
        "element_type": hit[2] if hit else None,
    })
print(f"{doc.title}  ({doc.n_pages} pages)\n")
pd.DataFrame(rows)

Annual Report 2024-25  (326 pages)


,concept,oracle_cr,found_as,page,element_type
0,Gross Profit,"39,844",-- DISCARDED --,NaN,NaN
1,Net Income,"10,929","109,290",6.0,text
2,Operating Income,"12,581",-- DISCARDED --,NaN,NaN
3,Total Revenue,"52,041","520,412.5",259.0,table
